# 00 - Project overview and structure

**SDSS17 Astronomical Object Classification** - classify every object in the Sloan Digital Sky Survey (SDSS17) sample as a
`GALAXY`, a `QSO` (quasar) or a `STAR`, using its photometry (five brightness measurements) and its redshift.

This notebook is the *map* of the project: what each file does, in which order to run things, and which design decisions were made and why.
Use it as the script when you explain the project.

## 1. The question and the data
* **Input:** `data/star_classification.csv` - 100,000 objects, 18 columns (the file you uploaded; no other copy of the data is used).
* **Target:** `class` (GALAXY 59.4 %, STAR 21.6 %, QSO 19.0 %) - imbalanced, so the main metric is **Macro-F1** (every class counts equally).
* **Useful columns:** the five magnitudes `u g r i z` (brightness through five filters) and `redshift` (how fast the object recedes - the physical clue that best separates quasars/galaxies from stars).
* **Not used as inputs:** survey bookkeeping such as `obj_ID`, `run_ID`, `rerun_ID`, `cam_col`, `field_ID`, `spec_obj_ID`, `plate`, `MJD`, `fiber_ID`, and the sky position `alpha`, `delta`.
  They describe *when/where the survey observed*, not *what the object is*, so a model using them would learn survey artefacts instead of astrophysics.

## 2. Pipeline at a glance

```
star_classification.csv
        |
        v
01_EDA ------------------> cleaning rule, engineered colour indices, stratified 70/15/15 split, figures/tables
        |
        +--> 02_EMeans -----------> unsupervised clustering (Faiss E-Means, k=3) + 4 clustering metrics
        |
        +--> 03_CatBoost ---------> Optuna (validation Macro-F1) -> best CatBoost, with and without redshift
        |
        +--> 04_LightGBM ---------> Optuna (validation Macro-F1) -> best LightGBM, with and without redshift
                    |
                    v
              05_Comparison -------> ONE evaluation on the untouched test set, model selection, explanation,
                    |               final model saved + registered in MLflow
                    v
                  app.py ---------> Streamlit app that reads everything in artifacts/ and serves predictions
```
Every notebook rebuilds the *same* split from `src/common.py` (fixed seed 42), so all models are compared on identical train/validation/test rows.

## 3. Folder structure
| Path | What it contains |
|---|---|
| `notebooks/00_Project_Overview.ipynb` | this map |
| `notebooks/01_Data_Analysis_EDA.ipynb` | data inspection, cleaning, EDA, feature engineering, split |
| `notebooks/02_EMeans_Clustering.ipynb` | E-Means clustering experiment |
| `notebooks/03_CatBoost_Optuna_MLflow.ipynb` | CatBoost tuning/training/evaluation |
| `notebooks/04_LightGBM_Optuna_MLflow.ipynb` | LightGBM tuning/training/evaluation |
| `notebooks/05_Model_Comparison_and_Final_Model.ipynb` | comparison, selection, explanation, final model |
| `src/common.py` | small shared helpers: constants, cleaning, features, split, metrics, plotting, MLflow setup |
| `app.py` | Streamlit application (the interactive deliverable) |
| `artifacts/` | everything the notebooks produce: `eda/`, `emeans/`, `comparison/`, `models/`, `final_model.json`, `data_sample.csv`, `mlflow_runs.csv` |
| `mlflow.db`, `mlruns/` | MLflow tracking store (SQLite database + logged artifacts and models) |
| `data/` | the dataset |
| `run_notebooks.sh` | executes notebooks 01-05 in order |

## 4. Key design decisions (and the reason for each)
| Decision | Reason |
|---|---|
| Remove the one row with `-9999` in `u/g/z` | SDSS uses -9999 for "no measurement"; it is a missing value, not a magnitude. It needs no fitted statistic, so removing it before the split is leak-free. |
| Colour indices `u-g, g-r, r-i, i-z` | Differences of magnitudes measure the *shape* of the spectrum independent of overall brightness - the classic way astronomers separate object types. |
| Stratified 70/15/15 split, seed 42 | Keeps the class proportions identical in train/val/test; the test set is only used after tuning. |
| Scaler fit on train only (E-Means) | Prevents information from validation/test leaking into preprocessing. Tree models do not need scaling. |
| Faiss `E-Means` (not K-Means) | Required by the brief. Verified: `faiss.contrib.e_means.NumpyEmeans`, available in `faiss-cpu >= 1.16.0`. |
| Optuna objective = **validation** Macro-F1 | The test set must stay untouched during tuning and selection of hyper-parameters. |
| Early stopping on the validation set | Picks the number of trees automatically; still no test data involved. |
| Compare with and without `redshift` | Quantifies how much of the performance is owed to that single feature. |
| Final model = highest **test** Macro-F1 | As specified; a paired bootstrap tells us whether the winner is really better or just lucky. |

## 5. How to run
```bash
pip install -r requirements.txt
./run_notebooks.sh                 # runs 01 -> 05 (about 30 min on one CPU core; most of it is CatBoost tuning)
streamlit run app.py               # interactive application
mlflow ui --backend-store-uri sqlite:///mlflow.db
```
The Optuna budget is set by `N_TRIALS` / `N_TRIALS_NOZ` at the top of notebooks 03 and 04.

## 6. Live view of the project folder
The cell below lists what is currently on disk (it will look empty before the notebooks have been executed).

In [1]:
import sys
sys.path.insert(0, "../src")
from common import ROOT
for p in sorted(ROOT.rglob("*")):
    rel = p.relative_to(ROOT)
    if any(part in {"mlruns", "__pycache__", ".ipynb_checkpoints"} for part in rel.parts): continue
    if len(rel.parts) <= 3 and not (p.is_file() and rel.parts[0] == "artifacts" and len(rel.parts) == 3 and p.suffix in {".png", ".csv", ".json"} and rel.parts[1] in {"comparison"}):
        print("  " * (len(rel.parts) - 1) + p.name + ("/" if p.is_dir() else ""))

README.md
app.py
artifacts/
  comparison/
    with_redshift_catboost/
    with_redshift_lightgbm/
    without_redshift_catboost/
    without_redshift_lightgbm/
  data_sample.csv
  eda/
    boxplots_outliers.png
    class_distribution.png
    class_medians.csv
    color_color_redshift.png
    color_distributions.png
    correlation.png
    describe.csv
    feature_distributions.png
    kruskal_by_class.csv
    outliers_iqr.csv
    overview.json
  emeans/
    centroids_original_units.csv
    centroids_with_redshift.npy
    cluster_composition.png
    cluster_vs_class_counts_test.csv
    clusters_pca.png
    k_sweep.csv
    k_sweep.png
    scaler_with_redshift.joblib
    summary.json
  final_model.json
  mlflow_runs.csv
  models/
    with_redshift_catboost.cbm
    with_redshift_lightgbm.joblib
    without_redshift_catboost.cbm
    without_redshift_lightgbm.joblib
data/
  star_classification.csv
mlflow.db
notebooks/
  00_Project_Overview.ipynb
  01_Data_Analysis_EDA.ipynb
  02_EMeans_Clust